# CS383: Data Science and Machine Learning
## Lecture 10 — Tree-Based Models: Decision Trees, Random Forests, Gradient Boosting, and Regression Trees

*Dr. Thitima Srivatanakul*

### Guiding question
**You've built two very different classifiers -- one that fits a straight-line formula, one that looks
at nearby neighbors. What if a model could just ask a series of yes/no questions instead -- and could
that same idea predict a number, not just a category?**

### Learning objectives
By the end of this lecture, you should be able to:

**Must understand (the core of today):**

- explain how a decision tree makes a prediction by asking a sequence of yes/no questions about features
- explain how Gini impurity guides splitting: the tree prefers questions that leave purer (less mixed) groups
- explain why a single, unrestricted decision tree can overfit, check that with a train/test comparison
  (not training accuracy alone), and describe how limiting its depth trades fit for generalization
- explain how a random forest combines many different trees, and why that often makes predictions more
  stable than a single tree
- fit tree-based models with scikit-learn's familiar `.fit()` / `.predict()` interface (doing every learned
  preprocessing step after the train/test split), then evaluate and compare them on unseen data with accuracy,
  precision, recall, F1, and a confusion matrix, and explain why accuracy alone can mislead

**Introduction / extension (meet these today; you don't need to master them yet):**

- explain, at a high level, how gradient boosting builds trees one after another (each helping improve the
  current model), what XGBoost is, and why it doesn't always beat a simpler model
- read feature importances carefully: they show what a model *uses*, not what *causes* an outcome, and data
  leakage can make a feature look more important than it really is
- explain how a regression tree predicts a number (the mean of its leaf) and compare it with linear
  regression on the same data

---

### Where this fits
Lecture 8 gave you your first two classifiers -- logistic regression's straight-line boundary and
k-NN's local, neighbor-based one. Lecture 9 gave you the tools to judge any classifier fairly. Today
adds a third, very different way to draw a decision boundary: a tree of yes/no questions. Part 1 builds
a single decision tree on Lecture 8's toy reinspection dataset and visualizes its boundary next to the
straight line and the curvy neighbor-based one you've already seen. Part 2 scales that idea up to a
random forest -- many trees, combined. Part 3 tests tree-based models on the same real restaurant
inspections data from Lectures 6-9, with short introductions to gradient boosting (via XGBoost) and feature
importance. Part 4 closes the loop with Lecture 7: the same tree idea, used to predict a number instead of
a category, compared against linear regression.

All four parts are covered in class. **Parts 1 and 2 are the core** (decision trees, Gini impurity, overfitting,
random forests) and the ideas to master. Part 3 applies them to real data, and the boosting, feature-importance,
and regression-tree pieces are shorter introductions.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. You'll play "20 Questions" against a small set of restaurants: pick the
best first question for a decision tree (and see the Gini math behind "best"), explore how tree depth
changes the boundary and the accuracy on data the tree hasn't seen, watch a small random forest vote,
and step through gradient boosting one tree at a time.

**[Open the "Twenty Questions" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect10/trees_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Decision Trees

Same toy dataset as Lecture 8, Part 2: 40 restaurants that failed an inspection and were scheduled for a
reinspection, described by two features -- violations racked up, and days until the reinspection. You've
already seen logistic regression draw a single straight line through this data, and k-NN draw a boundary
that hugs individual points. This time: what if the classifier just asked yes/no questions about the
features, one at a time?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

VIOLATIONS = np.array([2, 0, 9, 8, 2, 9, 2, 8, 5, 3, 5, 0, 5, 2, 5, 3, 2, 5, 2, 5,
                       7, 0, 5, 6, 0, 6, 6, 5, 8, 10, 4, 10, 9, 7, 0, 10, 4, 0, 8, 6], dtype=float)
DAYS_UNTIL_REINSPECTION = np.array([258, 228, 273, 47, 323, 302, 322, 24, 256, 37, 24, 218, 24, 8, 341,
                                     114, 184, 229, 99, 310, 6, 351, 296, 374, 377, 137, 365, 104, 70,
                                     259, 155, 324, 225, 321, 167, 128, 298, 344, 347, 340], dtype=float)
PASSES_REINSPECTION = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
                                 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0])

X_toy = np.__________([VIOLATIONS, DAYS_UNTIL_REINSPECTION])
print(f"{len(PASSES_REINSPECTION)} restaurants, {PASSES_REINSPECTION.sum()} passed reinspection")


def plot_decision_boundary(model, X, y, title, ax=None):
    show = ax is None          # standalone call: draw on the current axes and show the figure
    if ax is None:
        ax = plt.gca()
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 20, X[:, 1].max() + 20
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    grid_preds = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    grid_preds = grid_preds.reshape(xx.shape)

    ax.contourf(xx, yy, grid_preds, levels=[-0.5, 0.5, 1.5], colors=["#F7D9D6", "#D6E4F0"], alpha=0.8)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm_r", s=70, edgecolor="white", linewidth=1)
    ax.set_xlabel("Violations")
    ax.set_ylabel("Days until reinspection")
    ax.set_title(title)
    if show:
        plt.show()

### How a decision tree decides

A decision tree makes a prediction by asking one yes/no question about a feature, following the branch
that matches the answer, and repeating -- until it lands on a leaf that just says "pass" or "fail." Fitting
a tree means choosing which question to ask at each step: the one that splits the training data into the
two most "pure" groups possible (as close to all-pass or all-fail as it can manage). No formula, no
probabilities required -- just a flowchart, built automatically from the data.

In [ ]:
shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=383)
shallow_tree.__________(X_toy, PASSES_REINSPECTION)

print(export_text(shallow_tree, feature_names=["violations", "days_until_reinspection"], show_weights=True))

Read that top to bottom like a flowchart. The first question is: "is `days_until_reinspection` at most 161?"
If yes, go left. That branch is already a leaf and predicts fail, and all 14 training restaurants that
landed there did fail. If no, go right and ask a second question: "is `violations` at most 3?" If yes, the
leaf predicts pass (all 10 restaurants there passed). If no, it predicts fail (10 failed, 6 passed). Follow
any restaurant's two numbers down the tree and you land on a leaf that predicts pass or fail. The
`weights: [fails, passes]` on each leaf are those training counts. Every branch is a single yes/no question
about a single feature -- that's the entire model.

#### First, what is Gini impurity?

To choose a good question, a tree needs a way to score how **mixed** a group is. Compare two tiny groups of
restaurants:

| Group | Passed | Failed | p_pass | p_fail | Gini |
|---|---:|---:|---:|---:|---:|
| A | 4 | 0 | 1.0 | 0.0 | 1 - 1.0² - 0.0² = **0** |
| B | 2 | 2 | 0.5 | 0.5 | 1 - 0.5² - 0.5² = **0.5** |

**Gini impurity** measures how mixed the classes in a group are:

`Gini = 1 - p_pass² - p_fail²`

where `p_pass` and `p_fail` are the fractions of the group that passed and failed.

- **Gini = 0** means the group contains only one class (perfectly pure), like Group A.
- **A higher Gini** means the classes are more mixed.
- **With two classes, the maximum is 0.5**: an even 50/50 split, like Group B.

You don't need to memorize the formula. The idea to keep is: *the tree prefers questions that leave
groups less mixed.* **Quick check:** what is the Gini of a group with 3 passed and 1 failed? (`p_pass = 0.75`,
`p_fail = 0.25`.)

#### Why that first question? A worked Gini example

Now the real data. How did the tree choose "is `days_until_reinspection` at most 161?" over every other
possible question? It tries a candidate question, looks at the two groups the question creates, and scores
how mixed they are with the Gini formula above. The question that leaves the purest groups wins.

Before any question, all 40 restaurants (16 passed, 24 failed) have Gini = 1 - 0.4² - 0.6² = **0.480**.
For the question "days until reinspection <= 161":

| Group | Restaurants | Passed | Failed | Gini |
|---|---:|---:|---:|---:|
| Yes (<= 161 days) | 14 | 0 | 14 | 1 - 0² - 1² = **0.000** |
| No (> 161 days) | 26 | 16 - 0 = 16 | 10 | 1 - (16/26)² - (10/26)² = **0.473** |

A question creates *two* groups, but we need a single score for the question. So we take a **weighted
average** of the two groups' Gini values, weighting each group by its share of the 40 restaurants. Bigger
groups count more. Without weights, a question that peels off one pure restaurant and leaves a messy group of
39 would get a great-looking score even though it barely helped. Here:
`(14/40) × 0.000 + (26/40) × 0.473 = 0.308`. That is lower than the 0.480 we started with, and lower than
every other first question the tree could have asked on this data, so it wins. In plain words: a good
question leaves groups with **fewer mixed pass/fail labels**. The tree repeats the same comparison inside
each group, one question at a time.

In [ ]:
plot_decision_boundary(shallow_tree, X_toy, PASSES_REINSPECTION, "Decision Tree Boundary (max_depth=2)")

Compare this to Lecture 8's two boundaries on this exact same data: logistic regression drew one
straight line; k-NN drew a boundary that curves smoothly around individual points. This one is made of
rectangles with straight, axis-aligned edges -- because every single split tests just one feature against
one cutoff, the boundary can only ever bend at a right angle, never diagonally or smoothly. That blocky
look isn't a limitation of this particular tree; it's what *every* decision tree boundary looks like.

### What happens with no depth limit?

`max_depth=2` only allowed the tree to ask two questions before committing to an answer. What if it could
ask as many questions as it wanted, splitting all the way down until every leaf is pure?

**Predict first:** if we allow a decision tree to keep asking more and more questions, will it necessarily
perform better on new data? Jot down a guess (yes / no / it depends) before you run the next cell.

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=383)  # no max_depth -- grows until every leaf is pure
deep_tree.fit(X_toy, PASSES_REINSPECTION)

print(f"Tree depth: {deep_tree.get_depth()}")
print(f"Training accuracy: {deep_tree.__________(X_toy, PASSES_REINSPECTION):.3f}")

plot_decision_boundary(deep_tree, X_toy, PASSES_REINSPECTION, "Decision Tree Boundary (unrestricted depth)")

100% training accuracy, and a boundary carved into lots of small rectangles that chase individual points -- the
same warning sign as k-NN with $k=1$ back in Lecture 8. An unrestricted tree keeps asking questions until
every training point is isolated into its own pure leaf. Perfect *training* accuracy by itself doesn't prove
the model is bad, though; the real question is whether it still does well on data it hasn't seen. `max_depth`
(and a handful of similar settings you'll meet again in Lecture 11) is how you rein the tree in --
deliberately stopping it from asking every possible question. Note that `max_depth` is a *maximum*: some
branches stop earlier when a group is already pure, so a depth-2 tree can have 3 leaves, not 4.

#### Does training accuracy tell the whole story?

The boundaries above were all fit on all 40 restaurants, so the accuracy numbers were *training*
accuracy. Let's do what we do on every real dataset: split first, fit on the training rows only, and compare
training accuracy with accuracy on restaurants the tree has never seen, for `max_depth` = 1, 2, 3, 4, and no limit.
With only 40 restaurants the test set is tiny (12 restaurants), so treat the exact numbers as illustrative.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_toy, PASSES_REINSPECTION, test_size=0.3, random_state=383, stratify=PASSES_REINSPECTION
)
print(f"{len(y_tr)} training restaurants, {len(y_te)} test restaurants\n")

depth_results = {}   # depth -> (train accuracy, test accuracy), reused below
print("max_depth   leaves   train accuracy   test accuracy")
for depth in [1, 2, 3, 4, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=383).fit(X_tr, __________)
    depth_results[depth] = (t.score(X_tr, y_tr), t.score(X_te, y_te))
    print(f"{str(depth):>9}   {t.get_n_leaves():>6}   {t.score(X_tr, y_tr):>14.3f}   {t.score(X_te, y_te):>13.3f}")

#### Seeing the depths side by side

The table gives the numbers; now let's *look* at them. Below are the decision boundaries for `max_depth` = 1, 2, 3, and 4
on the same data, the same training/test split, and the same axes and colors. Blue = passed reinspection and red = failed (the same
colors as before). Circles are restaurants, and the larger **diamonds with a black outline are the 12 held-out test
restaurants**. Each title shows that tree's
training and test accuracy.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4), sharex=True, sharey=True)

for ax, depth in zip(axes, [1, 2, 3, 4]):
    t = DecisionTreeClassifier(max_depth=depth, random_state=383).fit(X_tr, y_tr)
    train_acc = depth_results[depth][0]
    test_acc = t.score(X_te, __________)
    plot_decision_boundary(t, X_toy, PASSES_REINSPECTION,
                           f"max_depth={depth}\ntrain {train_acc:.2f} | test {test_acc:.2f}", ax=ax)
    # Re-draw the test restaurants as diamonds so they stand out from the training circles.
    ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap="coolwarm_r", vmin=0, vmax=1,
               marker="D", s=90, edgecolor="black", linewidth=1.2)

plt.tight_layout()
plt.show()

**Discuss with a neighbor:**

1. How do the decision boundaries change as tree depth increases?
2. Does a more complicated boundary necessarily improve test accuracy?

#### Try It: Does a Deeper Tree Always Perform Better?

Use the four plots and the table above (no new code needed):

1. Compare the decision boundaries for depths 1-4. What changes from one plot to the next?
2. Which depth has the **highest training accuracy**?
3. Which depth has the **highest test accuracy in this particular experiment**? (Is there a tie?)
4. Does the most complex tree give the best test performance? Explain.
5. Why could a different train/test split give different results?

When you have your answers, run the next cell to check what the code says.

In [ ]:
# Check your answers: everything printed here is computed from the results above.
shown = {d: depth_results[d] for d in [1, 2, 3, 4]}
top_train = max(a for a, _ in shown.values())
top_test = max(b for _, b in shown.values())

print("Highest training accuracy at max_depth =", [d for d, (a, _) in shown.items() if round(a, 6) == round(top_train, 6)])
print("Highest test accuracy at max_depth =    ", [d for d, (_, b) in shown.items() if round(b, 6) == round(top_test, 6)])

step = 100 / len(__________)   # percentage points that ONE test restaurant is worth
print(f"\nWith {len(y_te)} test restaurants, one restaurant changes test accuracy by about {step:.1f} percentage points.")

unlimited = DecisionTreeClassifier(random_state=383).fit(X_tr, y_tr)
print(f"With no depth limit, the tree grows to depth {unlimited.get_depth()} on these {len(y_tr)} training restaurants.")

A deeper tree can fit the training data more closely, but this does not guarantee better performance on unseen
data. Because our test set is small, the results are illustrative rather than conclusive.

- **Deeper trees can fit the training data more closely.** Training accuracy can only go up (or stay flat) as the
  tree is allowed more questions, because it keeps carving the training restaurants into smaller groups. That is
  why `max_depth` is a *ceiling*, not a target: the unrestricted tree stops at the depth printed in the check cell.
- **Higher training accuracy does not guarantee better test accuracy.** A tree can reach 100% on the training
  data by memorizing quirks that don't carry over to restaurants it hasn't seen.
- **Don't choose a depth from this one small test set.** With so few test restaurants, a single restaurant moves the
  score by the amount printed above, so a difference of one or two restaurants isn't strong evidence for any
  particular depth. A more reliable approach is to compare depths on validation data, or with cross-validation on
  the training set, and keep the test set for a final check. (Lecture 11 shows how.)

The lesson isn't "deeper is always worse"; it's that a high training score tells you the model fit the data it
saw, and only held-out data tells you whether it learned something that carries over.

---

## Part 2 — Random Forests: Many Trees, One Vote

A single deep tree can overfit, and a single shallow tree might underfit -- two questions isn't much to
work with. Single trees are also *unstable*: change a few training rows and you can get a quite different
tree. Instead of picking one tree and hoping it is exactly right, a **random forest** grows many trees and
lets them vote:

1. build many trees (often 100 or more), each on a random *bootstrap sample* of the training data (drawn with
   replacement, so no two trees see quite the same rows)
2. at each split, each tree is only allowed to consider a random subset of the features
3. for a new point, every tree makes its own prediction and the forest combines them -- think of it as a vote,
   where the most popular class wins

Each individual tree is still free to grow deep and overfit its own slice of the data -- but because every
tree sees different rows and features, their individual quirks tend to differ, and averaging their predictions
often reduces that instability (statisticians call it *variance*). "Often" is doing real work in that
sentence: a forest can still be wrong, and it is not guaranteed to beat a single tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=__________, random_state=383)
forest.fit(X_toy, PASSES_REINSPECTION)

plot_decision_boundary(forest, X_toy, PASSES_REINSPECTION, "Random Forest Boundary (100 trees)")

# Same forest, but shade each spot by its estimated probability of "pass" (the average of the 100 trees' estimates)
xx, yy = np.meshgrid(np.linspace(-1, 11, 300), np.linspace(-14, 398, 300))
prob_pass = forest.predict_proba(np.column_stack([xx.ravel(), yy.ravel()]))[:, 1].reshape(xx.shape)

plt.contourf(xx, yy, prob_pass, levels=np.linspace(0, 1, 11), cmap="Blues", alpha=0.85)
plt.colorbar(label="Estimated probability of pass (average over trees)")
plt.scatter(X_toy[:, 0], X_toy[:, 1], c=PASSES_REINSPECTION, cmap="coolwarm_r", s=70, edgecolor="white", linewidth=1)
plt.xlabel("Violations")
plt.ylabel("Days until reinspection")
plt.title("Random Forest: Estimated Probability of Passing")
plt.show()

Still boxy -- it's built from the same kind of yes/no splits, so the class boundary in the first plot isn't
dramatically smoother than a single deep tree's. The difference shows up in the second plot: instead of a hard
pass/fail, each spot is shaded by the forest's **estimated probability of passing**. Each tree estimates class
probabilities, and the random forest **averages** these estimates across its 100 trees. Near the middle of a
region the trees agree and the probability is close to 0 or 1; where they disagree, the shading fades gradually.
(The vote is a good way to think about the forest's final decision, but the probability shown here is an average
of the trees' probability estimates, so it is not simply the fraction of trees that predicted "pass.")

That's the same story as k=1 vs. k=15 in Lecture 8, arrived at a different way: instead of asking one model to
look at more neighbors, a random forest trains *many* different models and combines their predictions.
`n_estimators` -- how many trees to grow -- is the main dial: more trees generally means a more stable result,
at the cost of more computation. (Note this forest still scores 100% on its own training points; as with a
single tree, only held-out data can tell you how well it generalizes.)

**Technical Note (Optional Reading): where each tree's probability comes from**

In scikit-learn, a fitted tree estimates class probabilities from the class proportions in the leaf a new point
lands in. For example, a leaf holding 10 failed and 6 passed restaurants (like one leaf of the depth-2 tree earlier)
estimates 0.375 for "pass." A fully grown tree's leaves are usually pure, so a single deep tree's probabilities are
almost always exactly 0 or 1. The forest's `predict_proba()` averages these per-tree estimates, which is why it is
generally not the same as the fraction of trees that made a hard "pass" prediction, and why `predict()` (the class
with the highest averaged probability) can occasionally differ from a simple majority vote.

---

## Part 3 — Tree Models on Real Data (with Boosting and Feature Importance as Extensions)

So far we've met two ways of using trees. Here is the new one, side by side:

- **Random forest:** builds many randomized trees **independently** and combines their predictions.
- **Gradient boosting:** builds trees **one after another**, where each new small tree helps improve the
  current model. It starts with a simple prediction, and every new tree adds a small correction.
- **XGBoost:** a fast, heavily optimized implementation of gradient-boosted trees, with engineering features that
  make it quick on large tables. It also includes techniques such as **regularization** to control model complexity
  and reduce overfitting. We will discuss regularization in more detail in Lecture 11. XGBoost is one of the most
  widely used tools for tabular (rows-and-columns) data, and it follows the same `.fit()` / `.predict()` interface
  you already know.

Gradient boosting and XGBoost are a short introduction today, and you don't need the details. One thing to
watch for: **a fancier model is not automatically a better one.** Adding more trees does not guarantee better
accuracy on new data, and on small or low-signal datasets XGBoost often does no better than a simple tree.

Let's test all three tree-based approaches -- and feature importance -- on the real restaurant
inspections data from Lectures 6-9.

### Setup — NYC restaurant inspections

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
if not os.path.exists(raw_path):
    raise FileNotFoundError(
        f"Could not find the restaurant inspections snapshot at {raw_path}. "
        "This section needs the real NYC data (the same file used in Lectures 6-9), so it will not "
        "continue with made-up data. Check that the shared folder is available in your environment "
        "and ask your instructor if the file is missing."
    )

inspections_df = pd.read_csv(raw_path)
inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
inspections_df = inspections_df.dropna(subset=["score", "boro", "cuisine_description"]).reset_index(drop=True)
inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

print(f"Loaded real NYC restaurant inspection data: {len(inspections_df):,} violation records")
print(f"is_critical balance: {inspections_df['is_critical'].mean():.3f}")

#### What are we predicting, and what is one row?

- **One row = one violation** cited during one inspection of one restaurant. A single inspection usually
  produces several rows (about 3 on average), and a restaurant can be inspected more than once.
- **The target, `is_critical`**: is *this violation* flagged as a critical violation (1) or not (0)?
- **The features**: `score` (the inspection's overall score, which is the *same number on every row of that
  inspection*), `boro`, and the restaurant's cuisine.

This is a **retrospective analysis**: we're asking how well these features describe which violations ended up
critical, after the inspection has happened. It is not a realistic forecasting task, for the reason explained at
the end of this part (`score` is not known until the violations are recorded).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = inspections_df[["score", "boro", "cuisine_description"]]
y = inspections_df["is_critical"]

# Split FIRST -- before anything is learned from the data. Rows from the same restaurant stay together
# (see the note below), so we split by restaurant id (camis) instead of by individual row.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=383)
train_idx, test_idx = next(splitter.split(X, y, groups=inspections_df["camis"]))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("Restaurants appearing in both train and test:",
      len(set(inspections_df["camis"].iloc[train_idx]) & set(inspections_df["camis"].iloc[test_idx])))

# Which cuisines are "common"? Decide from the TRAINING rows only...
top_cuisines = X_train["cuisine_description"].value_counts().nlargest(10).index

# ...then apply that same rule to both sets (anything not on the list becomes "Other").
for part in (X_train, X_test):
    part["cuisine_top"] = part["cuisine_description"].where(part["cuisine_description"].isin(top_cuisines), "Other")
X_train = X_train.drop(columns="cuisine_description")
X_test = X_test.drop(columns="cuisine_description")

majority_class = y_train.__________()[0]
majority_baseline = (y_test == majority_class).mean()
print("Common cuisines (from training rows):", list(top_cuisines))
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

#### Why the order matters (connecting back to Lectures 6 and 7)

Deciding "which ten cuisines are the most common" is a *learned* step -- it counts rows. If we count on the
whole dataset before splitting, the test records help decide which categories the model gets to see by
name, which is a small leak of test information into preprocessing (the same mistake as fitting a scaler on
all the data). So the order is: **split first, learn the rules (the top-10 list, the one-hot categories) from
the training rows only, then apply those fixed rules to the test rows.** Dropping rows with a missing score or
borough, and turning `critical_flag` into 0/1, are different: they follow a fixed rule row by row and learn
nothing from the data, so doing them before the split is fine. The `ColumnTransformer` below follows the same
pattern: `fit_transform` on training data, `transform` only on test data.

**Why split by restaurant, not by row?** Records from the same restaurant should not appear in both the
training and the test set, because the model could then be tested on a restaurant it has already seen, which
gives an overly optimistic score. The rows are not independent. Rows from the same inspection share the
same `score`, and a restaurant's rows share its borough and cuisine. With a plain random row split, about 91% of
the test rows come from an inspection that also has rows in the training set, so the model can "recognize" an
inspection it has already seen instead of generalizing to new ones, which makes test accuracy look slightly
better than it should. Splitting by restaurant (`GroupShuffleSplit` with `groups=camis`) keeps all of a
restaurant's rows on the same side, so the test set is made of restaurants the model has never seen. (Here the
accuracies barely move, because there is so little signal to memorize. Splitting by group is still the right
habit.)

One setup step you *won't* see below: scaling `score`. A tree's split -- "is `score` at most 12.5?" --
only cares about the order of values, not their units or range, so scaling changes nothing about where a
tree decides to split. That's a real, practical difference from k-NN back in Lecture 8, which needed
`score` on the same footing as everything else because it measures raw distance. The only preprocessing
these models still need is one-hot encoding `boro` and `cuisine_top`, since trees (like every model
here) still need numbers in, not text.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["boro", "cuisine_top"]),
], remainder="passthrough")

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.__________(X_test)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=383)
tree_model.fit(X_train_ready, y_train)

tree_pred = tree_model.predict(X_test_ready)
print(f"Decision tree accuracy: {accuracy_score(y_test, tree_pred):.3f}  (baseline: {majority_baseline:.3f})")

A single depth-4 tree, same ballpark as logistic regression's roughly 59% from Lecture 8 on these same
three features (that lecture used a plain random row split, so treat the comparison as approximate) -- a modest step up from the baseline, not a dramatic one. That's a useful data point
already: whatever's limiting these models isn't the *shape* of the decision boundary (straight line vs.
tree of rectangles), since both land in about the same place.

In [ ]:
# max_depth=4 here too -- bagging makes overfitting less severe, but it doesn't erase it, so
# an unrestricted forest can still do worse than a depth-limited one on a weak-signal problem like this.
forest_model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=383)
forest_model.__________(X_train_ready, y_train)

forest_pred = forest_model.predict(X_test_ready)
print(f"Random forest accuracy: {accuracy_score(y_test, forest_pred):.3f}  (baseline: {majority_baseline:.3f})")

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=383, eval_metric="logloss")
xgb_model.fit(X_train_ready, y_train)

xgb_pred = xgb_model.__________(X_test_ready)
print(f"XGBoost accuracy:       {accuracy_score(y_test, xgb_pred):.3f}  (baseline: {majority_baseline:.3f})")

Decision tree, random forest, and XGBoost all land in roughly the same narrow band above the baseline,
not far from logistic regression's number from Lecture 8 either. XGBoost, the most sophisticated of the three, is not clearly ahead of the single depth-4 tree: **XGBoost does not
automatically outperform simpler models**, especially on small or low-signal data. That's a genuinely useful,
if slightly deflating, finding: swapping in a more expressive model -- one that can carve curved, complicated
boundaries instead of a single straight line -- doesn't manufacture signal that isn't in `score`, `boro`,
and `cuisine_top` to begin with. More powerful models can squeeze out real, if modest, gains when there
are non-linear patterns or feature interactions a straight line can't capture; they can't invent
information the features never had. That's the same honest-signal lesson from Lecture 8's real-data
section, holding up under a much stronger set of models.

### Comparing the models fairly

Accuracy is one number. Let's put all three models, plus the majority-class baseline, on the **same held-out
test set** and add the metrics from Lecture 9. In this problem the **positive class is `is_critical = 1`**
(a critical violation). **Precision** asks: of the violation records the model *flagged* as critical, what share
really were? **Recall** asks: of the records that really *were* critical, what share did the model catch?
**F1** balances the two. We also show training accuracy, so you can see the gap between training and test. Every
setting here (`max_depth=4`, 200 trees, learning rate 0.1) was fixed in advance rather than tuned against the
test set; Lecture 11 shows how to tune settings properly with cross-validation.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

models = {
    "Decision tree (depth 4)": tree_model,
    "Random forest": forest_model,
    "XGBoost": xgb_model,
}

baseline_pred = np.full(len(y_test), majority_class)
rows = [{
    "model": "Majority-class baseline",
    "train accuracy": (y_train == majority_class).mean(),
    "test accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "F1": f1_score(y_test, baseline_pred, zero_division=0),
}]
for name, model in models.items():
    pred = model.__________(X_test_ready)
    rows.append({
        "model": name,
        "train accuracy": accuracy_score(y_train, model.predict(X_train_ready)),
        "test accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })

pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

for name, model in models.items():
    cm = __________(y_test, model.predict(X_test_ready))
    print(f"{name}  (rows = actual, columns = predicted; 1 = critical)")
    print(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"]), "\n")

print("XGBoost classification report:")
print(classification_report(y_test, xgb_model.predict(X_test_ready), target_names=["not critical", "critical"]))

**Read this table carefully -- it tells a more honest story than accuracy alone.**

- **The baseline predicts "critical" for every violation record.** That gives it perfect recall (1.000: it "catches"
  every critical violation) and a precision equal to the base rate (0.565), and an F1 of 0.722 that looks
  respectable even though the baseline never tells restaurants apart. A high F1 or recall can come from
  simply flagging everything.
- **The random forest does almost the same thing**: its recall is 0.982 because it labels about 95% of the test
  records (2,723 of 2,856) as critical, so its F1 (0.731) is barely above the baseline's even though it
  is a much more complicated model.
- **The decision tree and XGBoost are more willing to say "not critical"** (see the confusion matrices), so
  their recall is lower and their precision is a little higher than the baseline's. They gain 3-4 points of
  accuracy over the baseline, but still miss many critical cases and raise many false alarms.
- **Training and test accuracy are nearly equal for every model.** That suggests there is no obvious large
  generalization gap in this experiment. It does not guarantee that overfitting is completely absent, but the
  bigger limit here looks like how little signal `score`, `boro`, and `cuisine_top` carry, not memorization.

**Why accuracy alone can mislead:** imagine only 5% of violation records were critical. A model that always says
"not critical" would be 95% accurate and catch none of them. Here the classes are closer to balanced (56%
critical), so accuracy is less distorted, but the same lesson holds: look at precision, recall, and the
confusion matrix, and compare against the baseline, before deciding a model is useful. Which trade-off you
prefer (catching more critical cases vs. fewer false alarms) depends on the real-world cost of each mistake.

### Which features is the model actually using? (Extension)

Every tree-based model above chose, at each split, which feature best separated the classes -- which
means it's kept a running tally of how useful each feature turned out to be. That tally is exposed as
`.feature_importances_`.

In [ ]:
feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(xgb_model.__________, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(7, 5))
importances.head(10).plot(kind="barh", color="#2B6CB0")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("XGBoost Feature Importances (top 10)")
plt.tight_layout()
plt.show()

importances.head(10)

`score` should sit at the top, clearly ahead of any single other column. That fits Lecture 8, where `score`
alone got about 59% accuracy and adding borough and cuisine moved it by only a fraction of a percent. A few
individual boroughs or cuisines (e.g. Manhattan, American) show up with smaller importance, each well below
`score`. But notice there are many of them: added together, the one-hot columns hold most of the model's
total importance, even though they barely improved accuracy.

Be careful what an importance number means:

- **It shows which features the model *uses*, not what *causes* the outcome.** A feature can matter to the
  model without having any real effect in the world.
- **Impurity- or split-based importance can be misleading.** It can favor features that offer many possible
  split points, like a numeric column such as `score`, over a yes/no column that can only be split one way. Two
  correlated features can also split the credit between them, which understates both. A different algorithm or
  train/test split can reorder the smaller importances while the big picture stays the same.
- **Data leakage can make a feature look important for the wrong reason.** Importance also says nothing about
  whether a feature would actually be *available* when you need to make a prediction -- which is the next
  question.

#### Could this model actually be used? Why `score` is a leakage risk

`score` is the strongest feature, but ask *when* it exists and *how* it is made. An inspection's score is
calculated from the violations the inspector finds during that same inspection, including the critical ones we
are trying to predict. So `score` is partly **built out of the target**: it isn't known before the violations are
recorded, and it already contains information about them. That is **target leakage**.

What that means for this lecture: this is a **retrospective** analysis. It shows how `score`, borough, and
cuisine relate to critical violations *after the fact*; it is **not** a realistic predictive task. You could not
use this model to forecast critical violations ahead of an inspection, and its ~60% accuracy (a few points above
the ~57% majority baseline) should not be read as an early-warning tool. For a genuinely predictive version you
would only use information available *before* the inspection, such as the restaurant's borough, cuisine, or past
inspection history. The habit to carry forward, from Lecture 7's leakage lesson: before trusting a model, ask what
it would actually know at the moment you need its prediction.

---

## Part 4 — Regression Trees: Predicting Numerical Values (Extension)

Everything so far predicted a **category** (pass/fail, critical/not). The same tree idea can predict a
**number**. This short section (about 8-10 minutes) connects trees back to Lecture 7: we take the same
**Study Hours vs. Exam Scores** data from Lecture 7's "What Is Regression, Really?" activity (12 students:
hours studied per week, and exam score), fit both a straight line and a regression tree, and compare them.
This part is self-contained: it doesn't use the restaurant data.

**Main idea:** classification trees predict **categories** and regression trees predict **numbers**. They share
the same structure -- a series of feature-based questions -- but use different splitting criteria. A regression
tree's prediction is a **staircase**: one constant value per range of the feature.

| | Classification tree | Regression tree |
|---|---|---|
| Predicts | A category | A numerical value |
| Example | Pass / fail | Exam score |
| Split criterion | Gini impurity | Squared-error reduction |
| Leaf prediction | Majority class | Mean target value |
| scikit-learn | `DecisionTreeClassifier` | `DecisionTreeRegressor` |

Both ask yes/no questions about features and end in a leaf. A classification tree picks the question that
makes the groups *purer*; a regression tree picks the question that makes each group's numbers *closer to
that group's average*, i.e. that reduces the squared prediction error.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# The exact 12-student dataset from Lecture 7's regression activity
HOURS = np.array([1, 2, 3, 3, 4, 5, 6, 6, 7, 8, 9, 10])
SCORES = np.array([50, 54, 60, 58, 65, 64, 72, 75, 78, 82, 88, 90])

X_h = HOURS.reshape(-1, 1)
Xh_train, Xh_test, yh_train, yh_test = train_test_split(X_h, SCORES, test_size=0.25, random_state=383)
print("Training hours:", sorted(Xh_train.ravel().tolist()), " Test hours:", sorted(Xh_test.ravel().tolist()))

linear_model = LinearRegression().fit(Xh_train, yh_train)
tree_reg = DecisionTreeRegressor(max_depth=2, random_state=383).fit(Xh_train, __________)

print(f"Linear regression: score = {linear_model.intercept_:.1f} + {linear_model.coef_[0]:.2f} x hours")
print(f"Regression tree: {tree_reg.get_n_leaves()} leaves, depth {tree_reg.get_depth()}")

Both models see exactly the same 9 training students and are scored on the same 3 held-out students. With
only 12 students in total, **the results below are illustrative**: change the random split and the numbers
(and even which model looks better) can change. We keep the split fixed (`random_state=383`) so everyone in
the room sees the same thing.

In [ ]:
x_grid = np.linspace(0, 12, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, model, title, color in [
    (axes[0], linear_model, "Linear regression: a straight line", "#D98C3F"),
    (axes[1], tree_reg, "Regression tree (max_depth=2): a staircase", "#2B6CB0"),
]:
    ax.plot(x_grid, model.__________(x_grid), color=color, linewidth=2.5, label="model prediction")
    ax.scatter(Xh_train, yh_train, color="#333333", s=50, label="training students")
    ax.scatter(Xh_test, yh_test, color="#E53E3E", s=70, marker="D", label="held-out students")
    ax.set_title(title)
    ax.set_xlabel("Study hours per week")
    ax.set_xlim(0, 12)
    ax.set_ylim(40, 100)
axes[0].set_ylabel("Exam score")
axes[0].legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"At 12 hours (beyond any student we saw):  linear = {linear_model.predict([[12]])[0]:.1f},  tree = {tree_reg.predict([[12]])[0]:.1f}")

**Linear regression** fits one straight line, so its prediction changes a little with every extra hour.
**The regression tree** asks threshold questions about hours, which cuts the hours axis into ranges, and it
predicts one constant value inside each range -- hence the staircase. Both axes use the same scales so the two
panels are directly comparable. One more difference shows at the far right: the tree has never seen more than 10
hours, so it just keeps predicting its last step, while the line keeps climbing. Trees can't extrapolate beyond
the range of the training data the way a line can (for better or worse).

In [ ]:
plt.figure(figsize=(9, 5))
__________(tree_reg, feature_names=["hours"], filled=True, rounded=True, precision=1)
plt.show()

# Check the leaf with 3 training students: is its prediction really their mean score?
leaf_ids = tree_reg.apply(Xh_train)
for leaf in np.unique(leaf_ids):
    scores_here = yh_train[leaf_ids == leaf]
    print(f"Leaf {leaf}: training scores {sorted(scores_here.tolist())} -> mean {scores_here.mean():.1f}  (tree predicts {tree_reg.tree_.value[leaf][0][0]:.1f})")

Read it like the classification trees: each internal node asks a numerical threshold question ("hours <= 5.0?"),
and each leaf holds a **number**. With squared-error splitting, a leaf's prediction is the **mean of the
training targets that reached it**. The printout shows this on the real tree: for example, the three training
students who land in the "more than 2 hours, at most 5 hours" leaf scored 58, 60, and 65, and
(58 + 60 + 65) / 3 = 61, exactly what the tree predicts there. The tree chose its thresholds by trying
candidate cut points and keeping the one that most reduced the total squared error around each group's mean.
(Don't mix this up with Gini impurity: Gini measures how mixed *classes* are; a regression tree has no
classes, so it measures squared distance from the average instead.)

In [ ]:
rows = []
for name, model in [("Linear regression", linear_model), ("Regression tree (depth 2)", tree_reg)]:
    pred = model.predict(Xh_test)
    rows.append({
        "model": name,
        "MAE": __________(yh_test, pred),
        "RMSE": np.sqrt(mean_squared_error(yh_test, pred)),
        "R2": r2_score(yh_test, pred),
    })
pd.DataFrame(rows).set_index("model").round(2)

These are the same three metrics from Lecture 7, computed on the same 3 held-out students for both models.
**Lower MAE** means a smaller average miss (in exam-score points). **Lower RMSE** means smaller errors once
squaring has put extra weight on the big misses. **Higher R²** means better performance than just predicting the
average score on this evaluation set. On this split the straight line comes out slightly ahead of the tree --
unsurprising, because exam scores here rise roughly in a straight line with study hours, which is exactly what
linear regression assumes. A regression tree earns its keep when the relationship is *not* a line (curves,
sudden jumps, thresholds), where a single straight line fits badly. With only 3 test students, don't read much
into a gap this small.

#### Check your understanding

1. **Why does the regression tree produce a staircase-shaped prediction instead of a straight line?**
2. **Does a more flexible regression tree necessarily perform better than linear regression on unseen data? Why or why not?**

**Extensions (not needed today):** the ensemble ideas from Parts 2 and 3 carry over directly --
`RandomForestRegressor` averages many regression trees, and gradient boosting also has a regression version
(`GradientBoostingRegressor`, `XGBRegressor`). We'll leave those for your own exploration.

---

## Check your understanding

Talk each one through with a neighbor (discussion questions, no calculation needed):

1. How does a decision tree make a prediction for a new observation?
2. Why does a decision tree prefer splits that create purer groups?
3. Why might a tree with 100% training accuracy perform poorly on test data?
4. How is a random forest different from a single decision tree?
5. Why doesn't XGBoost always outperform a simpler model?

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect10_tree_models_exercise.ipynb`.

---

## Cheat Sheet

| Task | Code |
|---|---|
| Fit a decision tree | `DecisionTreeClassifier(max_depth=d).fit(X_train, y_train)` |
| Read a tree as text | `export_text(model, feature_names=[...], show_weights=True)` |
| Fit a random forest | `RandomForestClassifier(n_estimators=n).fit(X_train, y_train)` |
| Fit an XGBoost classifier | `XGBClassifier(n_estimators=n, max_depth=d).fit(X_train, y_train)` |
| Fit a regression tree | `DecisionTreeRegressor(max_depth=d).fit(X_train, y_train)` |
| Feature importances | `model.feature_importances_` |
| Get one-hot feature names back | `preprocessor.get_feature_names_out()` |
| Accuracy | `accuracy_score(y_test, y_pred)` |
| Precision / recall / F1 | `precision_score`, `recall_score`, `f1_score(y_test, y_pred)` |
| Confusion matrix | `confusion_matrix(y_test, y_pred)` |
| Regression metrics | `mean_absolute_error`, `mean_squared_error`, `r2_score(y_test, y_pred)` |
| Majority-class baseline | `(y_test == y_train.mode()[0]).mean()` |
| Common categories from training rows | `X_train[col].value_counts().nlargest(k).index` |

---

## Key Terms

- **Decision tree**: a model that predicts by asking a sequence of yes/no questions about features,
  ending in a leaf that predicts a class; its boundary is always made of axis-aligned rectangles.
- **Overfitting (in a tree)**: an unrestricted tree keeps splitting until every training point is
  isolated, which can memorize the training set instead of learning the general pattern; you detect it by
  comparing training and test performance, and `max_depth` is one way to limit it.
- **Bagging**: training many models independently, each on a random bootstrap sample of the data, and
  combining (averaging or voting on) their predictions, which often makes the result more stable than any one model.
- **Random forest**: a bagged ensemble of decision trees, each trained on a random sample of rows and a
  random subset of features at each split. Each tree estimates class probabilities, and the forest averages
  them (majority vote is a good intuition for the final prediction).
- **Gradient boosting**: an ensemble built one tree at a time -- each new small tree helps improve the
  current model, rather than being trained independently like the trees in a random forest.
- **XGBoost**: a fast, optimized implementation of gradient-boosted trees (with built-in regularization, covered in Lecture 11), widely
  used for tabular data and following scikit-learn's `.fit()` / `.predict()` interface. Not automatically
  better than a simpler model.
- **Feature importance**: a score, per feature, of how useful that feature was for the splits a
  tree-based model actually made; a useful clue about the model, not a guarantee of real-world causal
  importance. It can favor features with many possible split points, and leakage can make a feature look
  more important than it really is.
- **Gini impurity**: a measure of how mixed the classes in a group are, `1 - p_pass² - p_fail²` for two classes
  (0 = all one class, 0.5 = an even split); a classification tree picks the question with the lowest size-weighted Gini.
- **Regression tree**: a decision tree that predicts a number; its splits reduce squared error and each
  leaf predicts the mean of the training targets that landed there.
- **Precision / recall / F1**: for a chosen positive class, the share of flagged cases that were truly
  positive, the share of truly positive cases that were caught, and a single score balancing the two.